# Generate the FP&A Dataset

This notebook creates `data/fpa_actuals_budget.csv`, the synthetic FP&A dataset used by
every lesson in this stage. The file is already included, so you only need to run this
notebook if you want to recreate it (run all cells, top to bottom).

The company, its product lines, and every number are **made up** with a fixed random
seed. There is no real company, customer, or employee data here.

## What is in the file

One row per month, product line, and region, from 2023-01 to 2025-12. Amounts are in
**USD thousands**.

| Column | Meaning |
|--------|---------|
| `Month` | Month as `YYYY-MM` |
| `ProductLine` | One of four fictional product lines |
| `Region` | North America, Europe, Asia Pacific, or Latin America |
| `Revenue`, `Budget_Revenue` | Actual and budgeted revenue |
| `COGS`, `Budget_COGS` | Actual and budgeted cost of goods sold |
| `OpEx`, `Budget_OpEx` | Actual and budgeted operating expenses |
| `Headcount` | Employees supporting that product line in that region |
| `FX_Rate` | Actual exchange rate divided by the budget rate (1.00 = as budgeted) |

In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("data", exist_ok=True)
rng = np.random.default_rng(2025)

months = pd.period_range("2023-01", "2025-12", freq="M")

# Monthly budget revenue for North America, gross margin, and OpEx as a share of revenue.
PRODUCT_LINES = {
    "Surgical Supplies":  dict(base=1800, margin=0.52, opex=0.24, staff=38),
    "Wound Care":         dict(base=1200, margin=0.61, opex=0.27, staff=26),
    "Diagnostics":        dict(base=950,  margin=0.58, opex=0.30, staff=31),
    "Monitoring Devices": dict(base=700,  margin=0.49, opex=0.33, staff=24),
}
# Size of each region relative to North America.
REGIONS = {"North America": 1.0, "Europe": 0.7, "Asia Pacific": 0.5, "Latin America": 0.3}

# Seasonality: a quieter summer and a strong fourth quarter.
SEASON = {1: 0.95, 2: 0.96, 3: 1.02, 4: 1.00, 5: 1.00, 6: 1.01,
          7: 0.93, 8: 0.92, 9: 1.01, 10: 1.04, 11: 1.06, 12: 1.10}

## The story in the numbers

Most months land close to budget. A few deliberate events give the lessons something to
explain (this list is the answer key, so trainees should not need to read it):

1. **Europe, 2025-04 to 2025-06:** the local currency weakens (FX rate about 0.92), so
   reported revenue for every product line falls below budget.
2. **Wound Care, Europe, 2025-05 to 2025-06:** a supply problem cuts volume by a further 12%.
3. **Diagnostics, North America, 2025-04 to 2025-06:** component costs rise, COGS about 12% over.
4. **Monitoring Devices, Asia Pacific, 2024-10 to 2024-12:** a large tender win, revenue about 25% over.
5. **Surgical Supplies, Latin America, 2025-09:** a one-off distributor transition, OpEx about 40% over.

In [ ]:
rows = []
for t, period in enumerate(months):
    month = str(period)
    growth = 1.06 ** (t / 12)          # 6% a year budgeted growth
    season = SEASON[period.month]
    for product, cfg in PRODUCT_LINES.items():
        for region, size in REGIONS.items():
            budget_revenue = cfg["base"] * size * growth * season
            budget_cogs = budget_revenue * (1 - cfg["margin"])
            budget_opex = budget_revenue * cfg["opex"]

            fx = 1.0 if region == "North America" else rng.normal(1.0, 0.015)
            volume = rng.normal(1.0, 0.03)
            cost_factor = rng.normal(1.0, 0.02)
            opex_factor = rng.normal(1.0, 0.025)

            # The deliberate events described above.
            if region == "Europe" and "2025-04" <= month <= "2025-06":
                fx = rng.normal(0.92, 0.005)
            if product == "Wound Care" and region == "Europe" and "2025-05" <= month <= "2025-06":
                volume *= 0.88
            if product == "Diagnostics" and region == "North America" and "2025-04" <= month <= "2025-06":
                cost_factor *= 1.12
            if product == "Monitoring Devices" and region == "Asia Pacific" and "2024-10" <= month <= "2024-12":
                volume *= 1.25
            if product == "Surgical Supplies" and region == "Latin America" and month == "2025-09":
                opex_factor *= 1.40

            revenue = budget_revenue * volume * fx
            # COGS follows units sold (volume) and local costs converted at the same FX rate.
            cogs = budget_cogs * volume * cost_factor * fx
            opex = budget_opex * opex_factor * fx

            rows.append({
                "Month": month,
                "ProductLine": product,
                "Region": region,
                "Revenue": round(revenue, 1),
                "Budget_Revenue": round(budget_revenue, 1),
                "COGS": round(cogs, 1),
                "Budget_COGS": round(budget_cogs, 1),
                "OpEx": round(opex, 1),
                "Budget_OpEx": round(budget_opex, 1),
                "Headcount": int(round(cfg["staff"] * size * growth ** 0.5 + rng.normal(0, 1))),
                "FX_Rate": round(fx, 3),
            })

df = pd.DataFrame(rows)
path = os.path.join("data", "fpa_actuals_budget.csv")
df.to_csv(path, index=False)
print(f"wrote {path}: {len(df)} rows, {df['Month'].min()} to {df['Month'].max()}")